[Reference](https://blog.gopenai.com/memory-in-ai-agents-why-your-agent-forgets-everything-and-how-to-fix-it-250150317ff1$0)

In [1]:
import json
from datetime import datetime
from openai import OpenAI

client = OpenAI()
# Our "memory store" — in production, this would be a real database
memory_store = {}
def extract_memories(conversation: str, user_id: str) -> list[str]:
    """Ask the LLM to pull out facts worth remembering long-term."""
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{
            "role": "user",
            "content": (
                "Extract any facts worth remembering long-term from this "
                "conversation — user preferences, account details, stated "
                "goals. Return a JSON list of short factual strings. "
                "If nothing is worth remembering, return an empty list.\n\n"
                f"Conversation:\n{conversation}"
            )
        }],
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content).get("facts", [])
def save_memories(user_id: str, facts: list[str]):
    """Store extracted facts against this user."""
    if user_id not in memory_store:
        memory_store[user_id] = []
    for fact in facts:
        memory_store[user_id].append({
            "fact": fact,
            "timestamp": datetime.now().isoformat()
        })
def get_relevant_memories(user_id: str) -> str:
    """Retrieve stored facts for this user (simplified — no semantic search yet)."""
    if user_id not in memory_store:
        return "No prior history with this user."
    facts = [m["fact"] for m in memory_store[user_id]]
    return "\n".join(f"- {f}" for f in facts)
def chat_with_memory(user_id: str, user_message: str) -> str:
    """A chat turn that both uses and updates long-term memory."""
    prior_context = get_relevant_memories(user_id)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": f"What you know about this user so far:\n{prior_context}"
            },
            {"role": "user", "content": user_message}
        ]
    )
    reply = response.choices[0].message.content
    # After responding, extract and save anything new worth remembering
    new_facts = extract_memories(f"User: {user_message}\nAssistant: {reply}", user_id)
    if new_facts:
        save_memories(user_id, new_facts)
    return reply
# Simulate two separate sessions
print(chat_with_memory("user_123", "Hey, I'm on the Pro plan and I prefer email updates."))
print("\n--- New session, different day ---\n")
print(chat_with_memory("user_123", "What's my current plan again?"))